In [72]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np

In [4]:
tickers = ["AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","JPM","XOM","KO"]
data = yf.download(tickers, start="2015-01-01", auto_adjust=False)

[*********************100%***********************]  10 of 10 completed


In [5]:
adj = data["Adj Close"].dropna(how="all")
adj.to_csv("adj_close.csv")

In [9]:
df = pd.read_csv('adj_close.csv', parse_dates = ['Date'], index_col = 'Date')

In [10]:
print(df.head)

<bound method NDFrame.head of                   AAPL        AMZN       GOOGL         JPM         KO  \
Date                                                                    
2015-01-02   24.237553   15.426000   26.278948   46.511139  29.783405   
2015-01-05   23.554739   15.109500   25.778227   45.067196  29.783405   
2015-01-06   23.556957   14.764500   25.142035   43.898651  30.009575   
2015-01-07   23.887281   14.921000   25.068090   43.965630  30.384165   
2015-01-08   24.805084   15.023000   25.155432   44.948105  30.751678   
...                ...         ...         ...         ...        ...   
2026-01-22  248.350006  234.339996  330.540009  303.630005  71.870003   
2026-01-23  248.039993  239.160004  327.929993  297.720001  72.879997   
2026-01-26  255.410004  238.419998  333.260010  301.040009  72.559998   
2026-01-27  258.269989  244.679993  334.549988  300.309998  73.550003   
2026-01-28  256.440002  243.009995  336.010010  300.769989  73.059998   

                  ME

In [11]:
df = df.sort_index()

In [12]:
df.dropna(how = 'all')

,AAPL,AMZN,GOOGL,JPM,KO,META,MSFT,NVDA,TSLA,XOM
Date,,,,,,,,,,
2015-01-02,24.237553,15.426000,26.278948,46.511139,29.783405,77.905800,39.858463,0.483011,14.620667,57.916897
2015-01-05,23.554739,15.109500,25.778227,45.067196,29.783405,76.654541,39.491913,0.474853,14.006000,56.332191
2015-01-06,23.556957,14.764500,25.142035,43.898651,30.009575,75.621765,38.912296,0.460457,14.085333,56.032700
2015-01-07,23.887281,14.921000,25.068090,43.965630,30.384165,75.621765,39.406670,0.459257,14.063333,56.600487
2015-01-08,24.805084,15.023000,25.155432,44.948105,30.751678,77.637672,40.565945,0.476533,14.041333,57.542580
...,...,...,...,...,...,...,...,...,...,...
2026-01-22,248.350006,234.339996,330.540009,303.630005,71.870003,647.630005,451.140015,184.839996,449.359985,133.639999
2026-01-23,248.039993,239.160004,327.929993,297.720001,72.879997,658.760010,465.950012,187.669998,449.059998,134.970001
2026-01-26,255.410004,238.419998,333.260010,301.040009,72.559998,672.359985,470.279999,186.470001,435.200012,134.839996


In [13]:
pxm = df.resample("M").last()

/var/folders/22/n4rqdwx117s71799081xqv3m0000gn/T/ipykernel_82078/2394869847.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pxm = df.resample("M").last()


In [100]:
retm = pxm.pct_change()

In [99]:
#to make signal matches monthly return so signal every month and the result they predict are at the same place
fwdret = retm.shift(-1)

In [98]:
#momentum Pt-1/Pt-12 - 1
mom = pxm.shift(1)/pxm.shift(12) - 1

In [97]:
#find low volatility
retd = df.pct_change()
vol60 = retd.rolling(60).std()
lowvol = -vol60.resample("M").last()

/var/folders/22/n4rqdwx117s71799081xqv3m0000gn/T/ipykernel_82078/3846973302.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  lowvol = -vol60.resample("M").last()


In [101]:
#zscore 
def findzscore(ser):
    ser2 = ser.clip(lower = ser.quantile(0.05), upper = ser.quantile(0.95))
    if ser2.std() == 0:
        return ser2*0
    ser = (ser2 - ser2.mean()) / ser2.std()
    return ser
volzscore = lowvol.apply(findzscore, axis = 1)
momzscore = mom.apply(findzscore,axis = 1)
zscore = 0.5 * volzscore + 0.5 * momzscore

In [66]:
# portfolio_rett =i∑ wt,i ⋅ rt,i
N = 5
w = pd.DataFrame(0.0, index = zscore.index, columns = zscore.columns)
for t in zscore.index:
    s = zscore.loc[t].dropna()
    if len(s) <= N:
        continue
    top = s.nlargest(N).index
    w.loc[t, top] = 1/N
portret = (w * fwdret).sum(axis = 1, min_count = 1)

In [68]:
#net of transaction costs
tc = 0.001
turnover = w.diff().abs().sum(axis = 1)
net = portret - tc * turnover

In [92]:
#g12=(NAVT**1/T)**12=NAVT**12/T  σannual≈σmonthly *sqr12  Sharpe=Rannual−Rf = 0/σannual
net2 = net.dropna()
nav = (net2 + 1).cumprod()
annret = nav.iloc[-1]**(12/len((nav.dropna())))-1
annvol = net2.std()*(12**0.5)
sharp = annret / annvol

In [95]:
runningmax = nav.cummax()
dd = nav / runningmax - 1
ddmax = dd.min()